In [ ]:
pip install aiogram

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 677.3/677.3 kB 9.0 MB/s eta 0:00:00


In [10]:
#название бота @gpt_learning_Mustafin_Bot
import asyncio
import logging
import pandas as pd
import openai
from aiogram import Bot, Dispatcher, types
from aiogram.filters.command import Command
from aiogram.utils.keyboard import ReplyKeyboardBuilder
from aiogram import F
from scipy import spatial
from openai import OpenAI
import os
from google.colab import userdata
from openai import OpenAI
import getpass
import tiktoken
import ast


# Включаем логирование
logging.basicConfig(level=logging.INFO)


API_TOKEN = '7653217656:AAHQyXR3Iyqtl-zqU_XxIkNFFmC8GwgcePo'

# Объект бота
bot = Bot(token=API_TOKEN)
# Диспетчер
dp = Dispatcher()


EMBEDDING_MODEL = "text-embedding-ada-002"
GPT_MODEL = "gpt-3.5-turbo"

embeddings_path = "https://storage.yandexcloud.net/ai-2025/FIFA_2034.csv"

try:
  openai = OpenAI(
    api_key = os.environ.get("OPENAI_API_KEY"),
  )
except:
  # Использование ключа API от VseGpt
  key = userdata.get('OPENAI_API_KEY')
  os.environ["OPENAI_API_KEY"] = key

  # Адрес сервера VseGpt
  base_url = "https://api.vsegpt.ru/v1"
  os.environ["OPENAI_BASE_URL"] = base_url
  openai = OpenAI(
    api_key = os.environ.get("OPENAI_API_KEY"),
  )

df = pd.read_csv(embeddings_path)

df['embedding'] = df['embedding'].apply(ast.literal_eval)

# Функция поиска
def strings_ranked_by_relatedness(
    query: str, # пользовательский запрос
    df: pd.DataFrame, # DataFrame со столбцами text и embedding (база знаний)
    relatedness_fn=lambda x, y: 1 - spatial.distance.cosine(x, y), # функция схожести, косинусное расстояние
    top_n: int = 100 # выбор лучших n-результатов
) -> tuple[list[str], list[float]]: # Функция возвращает кортеж двух списков, первый содержит строки, второй - числа с плавающей запятой
    """Возвращает строки и схожести, отсортированные от большего к меньшему"""

    # Отправляем в OpenAI API пользовательский запрос для токенизации
    query_embedding_response = openai.embeddings.create(
        model=EMBEDDING_MODEL,
        input=query,
    )

    # Получен токенизированный пользовательский запрос
    query_embedding = query_embedding_response.data[0].embedding

    # Сравниваем пользовательский запрос с каждой токенизированной строкой DataFrame
    strings_and_relatednesses = [
        (row["text"], relatedness_fn(query_embedding, row["embedding"]))
        for i, row in df.iterrows()
    ]

    # Сортируем по убыванию схожести полученный список
    strings_and_relatednesses.sort(key=lambda x: x[1], reverse=True)

    # Преобразовываем наш список в кортеж из списков
    strings, relatednesses = zip(*strings_and_relatednesses)

    # Возвращаем n лучших результатов
    return strings[:top_n], relatednesses[:top_n]


# с этой функцией мы уже знакомы
def num_tokens(text: str, model: str = GPT_MODEL) -> int:
    """Возвращает число токенов в строке для заданной модели"""
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

# Функция формирования запроса к chatGPT по пользовательскому вопросу и базе знаний
def query_message(
    query: str, # пользовательский запрос
    df: pd.DataFrame, # DataFrame со столбцами text и embedding (база знаний)
    model: str, # модель
    token_budget: int # ограничение на число отсылаемых токенов в модель
) -> str:
    """Возвращает сообщение для GPT с соответствующими исходными текстами, извлеченными из фрейма данных (базы знаний)."""
    strings, relatednesses = strings_ranked_by_relatedness(query, df) # функция ранжирования базы знаний по пользовательскому запросу
    # Шаблон инструкции для chatGPT
    message = 'Use the below articles on the 2022 Winter Olympics to answer the subsequent question. If the answer cannot be found in the articles, write "I could not find an answer."'
    # Шаблон для вопроса
    question = f"\n\nQuestion: {query}"

    # Добавляем к сообщению для chatGPT релевантные строки из базы знаний, пока не выйдем за допустимое число токенов
    for string in strings:
        next_article = f'\n\nWikipedia article section:\n"""\n{string}\n"""'
        if (num_tokens(message + next_article + question, model=model) > token_budget):
            break
        else:
            message += next_article
    return message + question


def ask(
    query: str, # пользовательский запрос
    df: pd.DataFrame = df, # DataFrame со столбцами text и embedding (база знаний)
    model: str = GPT_MODEL, # модель
    token_budget: int = 4096 - 500, # ограничение на число отсылаемых токенов в модель
    print_message: bool = False, # нужно ли выводить сообщение перед отправкой
) -> str:
    """Отвечает на вопрос, используя GPT и базу знаний."""
    # Формируем сообщение к chatGPT (функция выше)
    message = query_message(query, df, model=model, token_budget=token_budget)
    # Если параметр True, то выводим сообщение
    if print_message:
        print(message)
    messages = [
        {"role": "system", "content": "You answer questions about the 2022 Winter Olympics."},
        {"role": "user", "content": message},
    ]
    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0 # гиперпараметр степени случайности при генерации текста. Влияет на то, как модель выбирает следующее слово в последовательности.
    )
    response_message = response.choices[0].message.content
    return response_message

@dp.message(Command("start"))
async def cmd_start(message: types.Message):
    logging.info("Команда /start выполнена.")
    builder = ReplyKeyboardBuilder()
    await message.answer("Введите ваш вопрос!", reply_markup=builder.as_markup(resize_keyboard=True))

@dp.message(lambda message: True)
async def answer(message: types.Message):

    if message.text == '/help':
      await cmd_help(message)
    else:
      logging.info(f"Получено сообщение: {message.text}")
      response = ask(message.text, df, GPT_MODEL)
      await message.reply(response)

@dp.message(Command("help"))
async def cmd_help(message: types.Message):
    logging.info("Запрошена помощь командой /help.")
    await message.answer("Это страница 'help'!")
    await message.answer("1) Тематика: 2034 FIFA World Cup")
    await message.answer(f"2) Число тем: {len(df)}")
    q = "Which countries are set to host the 2034 FIFA World Cup?"
    await message.answer(f"3) Пример вопроса: {q}")
    await message.answer(f"Пример ответа: {ask(q, df, GPT_MODEL)}")

async def main():
    logging.info("Запуск бота...")
    await dp.start_polling(bot)

if __name__ == "__main__":
    asyncio.run(main())


#ask('Which countries are set to host the 2034 FIFA World Cup?')
